# Question Answering with LangChain, OpenAI and MultiQuery Retriever — my solution

**Goal:** a small RAG chatbot. Workplace documents are split into chunks, turned into embeddings (OpenAI)
and stored in **Elasticsearch**. For each question, `MultiQueryRetriever` lets the LLM rephrase the question
a few ways, searches with every version, merges the results, and the LLM answers from those chunks.

**My setup (differences from the lab template):**
- Elastic Cloud **Serverless** project of type *Vector Database*. Serverless projects have no *Cloud ID*,
  so we connect with the project's **Elasticsearch endpoint URL** (`es_url`) instead of `es_cloud_id`.
- Python 3.14 with current LangChain packages: the old `langchain<0.3` pins don't install here, so
  `MultiQueryRetriever` comes from `langchain_classic`, the loader from `langchain_community`.
- The lab's completion model (`gpt-3.5-turbo-instruct`) is no longer available → `ChatOpenAI(model="gpt-4o-mini")`.

## Step 0 — Speed fix for this PC

Python here tries IPv6 first for every new connection and waits ~10 s for it to fail.
Sorting the addresses IPv4-first makes each OpenAI / Elastic call fast again.

In [1]:
# Fix for this PC: try IPv4 before IPv6, otherwise every OpenAI call waits ~40 s
# (safe to run more than once: the real function is saved only the first time)
import socket
if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo
socket.getaddrinfo = lambda *args, **kwargs: sorted(
    socket._original_getaddrinfo(*args, **kwargs), key=lambda addr: addr[0] != socket.AF_INET
)

## Step 1 — Imports

The same building blocks as the lab, from today's package names:
- `OpenAIEmbeddings` turns text into vectors,
- `ElasticsearchStore` stores and searches those vectors in Elasticsearch,
- `ChatOpenAI` is the LLM (writes the question variants and the final answer),
- `MultiQueryRetriever` does the "rephrase → search several times → merge" trick.

In [2]:
import os
from getpass import getpass

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

## Step 2 — Connect to Elasticsearch

Two values from the Elastic project:
1. **Elasticsearch endpoint** — project page → *Application endpoints* → *Elasticsearch*. It's only an address, so it's written in the code.
2. **API key** — a secret. It's stored in a private **`.env`** file next to this notebook (`ELASTIC_API_KEY=...`).
   `.env` is listed in `.gitignore`, so the key never ends up on GitHub. `load_dotenv()` reads it into the environment.

The OpenAI key is already set on this PC (`OPENAI_API_KEY` environment variable).

`index_name` is the name of the "table" our chunks go into — the lab asks for a meaningful name.

In [3]:
from dotenv import load_dotenv

# The endpoint is just an address (not a secret), so it can live in the code.
ELASTIC_URL = "https://my-vectordb-project-c74b87.es.us-central1.gcp.elastic.cloud"

# The API key is a secret: it lives in the private .env file next to this notebook
# (.env is in .gitignore, so it never goes to GitHub).
load_dotenv()
ELASTIC_API_KEY = os.environ.get("ELASTIC_API_KEY") or getpass("Elastic API key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")

INDEX_NAME = "workplace-docs"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings,
)

# Quick connection check: asks Elasticsearch who it is
print(vectorstore.client.info()["version"])

{'number': '9.7.0', 'build_flavor': 'serverless', 'build_type': 'docker', 'build_hash': '0f8cabd008beb9ba3fa0a78a3528e52c96b677c0', 'build_date': '2026-10-27T16:38:40.654789858Z', 'build_snapshot': False, 'lucene_version': '10.5.2', 'minimum_wire_compatibility_version': '9.7.0', 'minimum_index_compatibility_version': '9.7.0'}


## Step 3 — Download the sample documents

Elastic's example dataset: **15 fictional company documents** (work-from-home policy, vacation policy,
sales strategy, code of conduct, …). It's one JSON file — a list of documents, each with `content`, `name`,
`summary`, `url`, `category`, `updated_at`. We save it locally as `temp.json` so the loader can read it.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

print(len(data), "documents")
for d in data:
    print("-", d["name"])

15 documents
- Work From Home Policy
- April Work From Home Update
- Wfh Policy Update May 2023
- Fy2024 Company Sales Strategy
- Company Vacation Policy
- Swe Career Matrix
- Sales Engineering Collaboration
- Intellectual Property Policy
- Code Of Conduct
- Office Pet Policy
- Performance Management Policy
- Sales Organization Overview
- Compensation Framework For It Teams
- Updating Your Tax Elections Forms
- New Employee Onboarding Guide


## Step 4 — Split the documents into passages (chunks)

- **`metadata_func`** (the lab's TODO): for every document, copy `name`, `summary`, `url`, `category` and `updated_at`
  into the chunk's **metadata**. The text itself goes into `page_content`; metadata travels along with it.
  Later the answer prompt shows `SOURCE: {name}` — that's where the name comes from.
- **`JSONLoader`** reads `temp.json`; `jq_schema=".[]"` means "each item of the list is one document", and
  `content_key="content"` says which field is the text.
- **`RecursiveCharacterTextSplitter.from_tiktoken_encoder`** cuts long texts into chunks measured in **tokens**.
  The lab asks for **800 tokens per chunk with 400 tokens overlap** (each chunk repeats half of the previous one,
  so a sentence on a border is never lost).

In [5]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400  # define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

print(len(docs), "chunks")
print(docs[0].metadata["name"], "|", docs[0].metadata["category"], "|", docs[0].metadata["updated_at"])
print(docs[0].page_content[:300])

C:\Users\Windows\AppData\Local\Temp\ipykernel_2120\1698271311.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


15 chunks
Work From Home Policy | teams | 2020-03-01
Effective: March 2020
Purpose

The purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 pandemic and beyond.
Scope

This policy applies to


**What we saw:** 15 documents → **15 chunks**. Every document here is shorter than 800 tokens, so none of them
had to be cut — each chunk is one whole document. With longer documents (e.g. a 50-page handbook) the same code
would produce many overlapping chunks per document.

## Step 5 — Store the chunks in Elasticsearch (bulk import)

`ElasticsearchStore.from_documents(...)` does two things for every chunk:
1. sends its text to OpenAI and gets back an **embedding** (a list of 1,536 numbers that captures its meaning),
2. stores text + metadata + embedding as one record in the Elasticsearch index `workplace-docs`.

Running this cell twice would store every chunk twice, so we first **delete the index if it already exists** —
that makes the cell safe to re-run.

Then we create the two remaining pieces:
- **`llm`** — `ChatOpenAI(model="gpt-4o-mini", temperature=0)` (the lab's `OpenAI()` completion model is retired);
  `temperature=0` makes its answers as repeatable as possible,
- **`retriever`** — `MultiQueryRetriever.from_llm(...)`: wraps the normal Elasticsearch search and uses the LLM
  to write extra versions of every question.

In [6]:
# Make the cell safe to re-run: start from an empty index
if vectorstore.client.indices.exists(index=INDEX_NAME):
    vectorstore.client.indices.delete(index=INDEX_NAME)

vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name=INDEX_NAME,
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
)

# Make the new records searchable right away, then count them
vectorstore.client.indices.refresh(index=INDEX_NAME)
print(vectorstore.client.count(index=INDEX_NAME)["count"], "chunks stored in", INDEX_NAME)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

15 chunks stored in workplace-docs


**What we saw (my test run):** 15 chunks stored in about 9 seconds.
You can also see them in Elastic: project → *Open project* → *Index Management* → `workplace-docs`.

## Step 6 — Question answering with the MultiQuery Retriever

The full RAG chain, piece by piece:

1. **`LLM_CONTEXT_PROMPT`** — the instructions for the final answer: *use only the retrieved context; if you don't know, say so*.
2. **`LLM_DOCUMENT_PROMPT`** — how each retrieved chunk is shown to the LLM: a `SOURCE: {name}` line (from our metadata) plus its text.
3. **`_combine_documents`** — glues all retrieved chunks into one block of text for `{context}`.
4. **`_context = RunnableParallel(...)`** — runs two things side by side on the same question:
   - `context`: retriever → combine chunks,
   - `question`: `RunnablePassthrough()` simply passes the question through unchanged.
5. **`chain = _context | LLM_CONTEXT_PROMPT | llm`** — fill the prompt, ask the LLM.

**Logging** is switched on for the retriever, so the output also shows the **alternative questions** the LLM generated.
(The logger name is `langchain_classic.retrievers.multi_query` in today's packages — the lab's `langchain.retrievers.multi_query` would stay silent.)

`llm` is a chat model, so the answer is a message object — we print its `.content`.

In [7]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, format_document

import logging

logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans.content)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What does the NASA sales team do and what are their responsibilities?  ', 'Can you provide information about the structure and functions of the NASA sales team?  ', 'Who are the members of the NASA sales team and what roles do they play?']


---- Answer ----
The NASA sales team refers to the sales organization that operates within the North America and South America region, as indicated in the context provided. This region is part of a larger sales organization structured to effectively serve customers and achieve business objectives across multiple regions.

In the context of the NASA sales team, it is led by two Area Vice-Presidents: Laura Martinez, who oversees North America, and Gary Johnson, who is responsible for South America. The team is composed of dedicated account managers, sales representatives, and support staff who work collaboratively to identify and pursue new business opportunities, nurture existing client relationships, and ensure customer satisfaction.

The NASA sales team plays a crucial role in understanding the unique market dynamics and customer needs within both North America and South America, allowing them to tailor their sales strategies and offerings accordingly. Their efforts are essential for 

**What we saw (my test run):**
- The log line *Generated queries* shows 3 rephrasings, e.g. *"What does the NASA sales team do and what are their responsibilities?"* and
  *"Who are the members of the NASA sales team and what roles do they play?"*.
- The answer: in these documents **NASA is not the space agency** — it is the company's **North America + South America** sales region,
  led by two Area Vice-Presidents (Laura Martinez for North America, Gary Johnson for South America).
  A plain LLM without retrieval could never know that — this is exactly what RAG is for: answers grounded in *your* documents.

<hr>

# 🚀 Your turn — 2 questions about the work-from-home policy

The dataset has **three** documents about working from home, and they **contradict each other** because the rule changed over time:

| Document | Date | Rule |
|---|---|---|
| Work From Home Policy | March 2020 | full-time work from home; company provides laptop, software licenses, secure communication tools |
| April Work From Home Update | April 2022 | from May 2022: **2 days a week in the office** |
| Wfh Policy Update May 2023 | May 2023 | from May 1, 2023: **3 days in the office, 2 days remote** |

So these are good test questions: the chatbot has to find the right documents **and** pick the most recent rule.

## Question 1 — How many days can I work from home?

**Correct answer (from the documents):** since May 2023, **2 days a week from home** (3 in the office).

In [8]:
ans = chain.invoke("How many days per week can employees work from home under the current policy?")

print("---- Answer ----")
print(ans.content)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What is the maximum number of days per week that employees are allowed to work remotely according to the current policy?', 'Under the existing policy, how many days each week are employees permitted to work from home?', 'Can you provide details on the weekly limit for remote work days for employees as specified in the current policy?']


---- Answer ----
Under the current work-from-home policy, which was updated on May 1, 2023, employees are required to work from the office three days a week, allowing for two days designated for remote work. This means that employees can work from home for a maximum of two days each week. It's important for employees to communicate with their supervisors and the HR department to establish their specific in-office workdays. This adjustment reflects the company's ongoing commitment to employee well-being while also ensuring collaboration and productivity in the workplace.


## Question 2 — What equipment do I get for working from home?

**Correct answer (from the documents):** a **company-issued laptop, software licenses and access to secure communication tools**;
employees must maintain and protect the equipment and data, and set up a safe, ergonomic workspace themselves.

In [9]:
ans = chain.invoke("What equipment does the company provide to employees who work from home?")

print("---- Answer ----")
print(ans.content)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What tools and resources does the company supply for remote employees?  ', 'What kind of equipment is available for employees who are working remotely?  ', 'Can you list the items that the company offers to support employees in a home office setup?']


---- Answer ----
The company provides several essential pieces of equipment and resources to employees who work from home to ensure they can perform their duties effectively and securely. Specifically, employees can expect to receive:

1. **Company-Issued Laptop**: This is typically a standard laptop that is configured with the necessary software and security features required for the employee's role.

2. **Software Licenses**: Employees will be provided with the necessary software licenses for applications and tools that are essential for their work. This may include productivity software, communication tools, and any specialized applications relevant to their job functions.

3. **Access to Secure Communication Tools**: To facilitate effective communication and collaboration with colleagues and supervisors, employees will have access to secure communication platforms. This ensures that discussions and data sharing remain confidential and protected.

It is important for employees to ma

## Reflection

*(Based on my test run — the generated questions can be worded a little differently each time.)*

**Were the generated questions useful?** Yes. For question 1 the LLM wrote versions like *"…how many days a week are employees permitted
to **telecommute**?"* and *"…the weekly limit for **remote work** days…"*. Those use words that are **not** in my question
("telecommute", "remote work"), so the search also matches documents that phrase things differently — that's the point of
`MultiQueryRetriever`. For question 2 it asked about *"tools and resources"* and *"home office equipment"* — again useful synonyms.
One small weakness: the three versions were quite similar to each other, so they added breadth but not much variety.

**Was the answer correct and grounded?**
- **Question 1 — correct.** The retriever found **all three** work-from-home documents (plus the vacation policy, which the LLM rightly ignored).
  Despite the contradictions, the answer was the **latest** rule: *3 days in the office, 2 days from home since May 2023*.
  It could only tell which rule is newest because the date is written inside the text ("Starting May 1, 2023"). Our
  `LLM_DOCUMENT_PROMPT` shows only `SOURCE: {name}` — adding `{updated_at}` from the metadata would make this more reliable
  if a document didn't mention its date.
- **Question 2 — correct.** Laptop, software licenses and secure communication tools, plus the duty to protect equipment and data —
  exactly the March 2020 policy. Because the prompt says *"be as verbose as possible"*, the answer added general explanations
  (e.g. what communication tools *may* include) that are not in the document — harmless here, but it shows the model can
  pad the answer beyond the retrieved context.

**Takeaway:** multi-query retrieval found every relevant document, and the answers stayed grounded. The main risks are
**outdated documents** (old and new rules sit side by side in the index) and the **"verbose"** instruction inviting extra,
unsourced detail.